<a href="https://colab.research.google.com/github/JacekDrzycimski/6tunnel/blob/master/Comms_3_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chat history container
chat_history = []

# Custom CSS injection for improved styling (Dark/Black Background Theme)
custom_css = """
<style>
    .chat-container {
        border: 1px solid #2c3e50;
        border-radius: 8px;
        padding: 15px;
        background-color: #121212;
        box-shadow: 0 4px 6px rgba(0,0,0,0.5);
        max-width: 600px;
    }
    .chat-history-box {
        background-color: #1e1e1e;
        border: 1px solid #333333 !important;
        border-radius: 6px;
        font-family: 'Courier New', Courier, monospace;
        padding: 10px !important;
        color: #00ff00;
    }
    .chat-title {
        color: #ffffff;
        font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
        margin-top: 0px;
        margin-bottom: 15px;
        border-bottom: 2px solid #3498db;
        padding-bottom: 5px;
    }
</style>
"""

# UI Elements
css_style = widgets.HTML(custom_css)
title_label = widgets.HTML("<h2 class='chat-title'>Simple Communications Client</h2>")
history_area = widgets.Output(layout={'border': 'none', 'height': '220px', 'overflow_y': 'auto'})

# Wrap history area inside an HTML box for applying custom CSS class
history_wrapper = widgets.VBox([history_area])
history_wrapper.add_class("chat-history-box")

username_input = widgets.Text(value='User', placeholder='Username', layout=widgets.Layout(width='25%'))
message_input = widgets.Text(placeholder='Type your message here...', layout=widgets.Layout(width='45%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='15%'))
clear_button = widgets.Button(description='Clear', button_style='warning', layout=widgets.Layout(width='15%'))

def update_history():
    with history_area:
        history_area.clear_output()
        for timestamp, sender, msg in chat_history:
            if sender != "Client-Bot":
                print(f"[{timestamp}] {sender}: {msg}")
            else:
                print(f"[{timestamp}] Bot: {msg}")
        # Auto-scroll JavaScript execution
        display(HTML("<script>var el = document.querySelector('.chat-history-box'); if (el) { el.scrollTop = el.scrollHeight; }</script>"))

def on_send_clicked(b):
    global chat_history
    msg_text = message_input.value.strip()
    username = username_input.value.strip() or 'User'
    if msg_text:
        user_timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        chat_history.append((user_timestamp, username, msg_text))
        message_input.value = ''

        bot_timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        chat_history.append((bot_timestamp, "Client-Bot", f"Received: '{msg_text}'"))

        # Keep only the last 10 messages
        chat_history = chat_history[-10:]
        update_history()

def on_clear_clicked(b):
    global df
    # Persist chat history to a local JSON file before clearing
    if chat_history:
        try:
            with open('chat_history_backup.json', 'w') as f:
                json.dump(chat_history, f, indent=4)
            print("Chat history successfully backed up to chat_history_backup.json")

            # Load into DataFrame globally
            df = pd.read_json('chat_history_backup.json')
            if not df.empty and len(df.columns) == 3:
                df.columns = ['Timestamp', 'Sender', 'Message']

            # Auto-plot message frequency
            with history_area:
                clear_output(wait=True)
                print("Generating Message Frequency visualization...")
                plt.figure(figsize=(6, 3.5))
                sns.countplot(data=df, x='Sender', palette='viridis')
                plt.title('Message Frequency by Sender')
                plt.xlabel('Sender')
                plt.ylabel('Number of Messages')
                plt.grid(axis='y', linestyle='--', alpha=0.7)
                plt.tight_layout()
                plt.show()

        except Exception as e:
            print(f"Failed to back up or plot chat history: {e}")
    chat_history.clear()
    if not 'df' in globals():
        update_history()

send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)
message_input.on_submit(on_send_clicked)

# Layout Arrangement
input_box = widgets.HBox([username_input, message_input, send_button, clear_button], layout=widgets.Layout(margin='10px 0px 0px 0px'))
ui_container = widgets.VBox([css_style, title_label, history_wrapper, input_box])
ui_container.add_class("chat-container")

display(ui_container)
update_history()

In [20]:
import pandas as pd
import os

file_path = 'chat_history_backup.json'

if os.path.exists(file_path):
    # Load the JSON data
    df = pd.read_json(file_path)
    # Add readable column names since the JSON stores it as a list of lists/tuples
    if not df.empty and len(df.columns) == 3:
        df.columns = ['Timestamp', 'Sender', 'Message']
    display(df)
else:
    print(f"The file '{file_path}' does not exist yet. Please send some messages and click 'Clear' in the chat client above to generate the file first!")

The file 'chat_history_backup.json' does not exist yet. Please send some messages and click 'Clear' in the chat client above to generate the file first!


In [21]:
def export_to_csv(dataframe, output_filename='chat_history_export.csv'):
    """
    Exports the given DataFrame to a CSV file.
    """
    if dataframe is None or dataframe.empty:
        print("DataFrame is empty or None. Please ensure you have chat history to export.")
        return
    try:
        dataframe.to_csv(output_filename, index=False)
        print(f"Chat history successfully exported to '{output_filename}'!")
    except Exception as e:
        print(f"An error occurred while exporting to CSV: {e}")

# Example usage (commented out until you have a populated 'df'):
# export_to_csv(df)

In [22]:
# Safely export the loaded DataFrame to a CSV file if it exists
if 'df' in globals() and not df.empty:
    export_to_csv(df, output_filename='chat_history_export.csv')
else:
    print("The DataFrame 'df' is not defined or is empty.")
    print("Please send messages in the chat client, click 'Clear', and ensure Cell 4a5e4ef5 is executed.")

The DataFrame 'df' is not defined or is empty.
Please send messages in the chat client, click 'Clear', and ensure Cell 4a5e4ef5 is executed.


In [23]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_message_frequency(dataframe):
    if dataframe is None or dataframe.empty:
        print("The DataFrame is empty. Please make sure you have generated the backup and loaded it in the cells above!")
        return

    plt.figure(figsize=(8, 5))
    sns.countplot(data=dataframe, x='Sender', palette='viridis')
    plt.title('Message Frequency by Sender')
    plt.xlabel('Sender')
    plt.ylabel('Number of Messages')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Safely call the visualization function with the loaded data if it exists
if 'df' in globals():
    plot_message_frequency(df)
else:
    print("The variable 'df' is not defined yet. Please make sure to send some messages, click 'Clear' in the chat interface, and run Cell 4a5e4ef5 to load the data first.")

The variable 'df' is not defined yet. Please make sure to send some messages, click 'Clear' in the chat interface, and run Cell 4a5e4ef5 to load the data first.


In [24]:
if 'df' in globals() and not df.empty:
    print("### General Descriptive Statistics ###")
    display(df.describe(include='all'))

    print("\n### Message Value Counts by Sender ###")
    display(df['Sender'].value_counts())

    # Calculate average message length
    df['Message_Length'] = df['Message'].astype(str).apply(len)
    print("\n### Average Message Character Length by Sender ###")
    display(df.groupby('Sender')['Message_Length'].mean().reset_index())
else:
    print("The DataFrame 'df' is empty or not defined. Please interact with the chat client first!")

The DataFrame 'df' is empty or not defined. Please interact with the chat client first!


In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if 'df' in globals() and not df.empty:
    # Convert Timestamp column to datetime object for proper plotting
    temp_df = df.copy()
    temp_df['Timestamp_parsed'] = pd.to_datetime(temp_df['Timestamp'], format='%H:%M:%S').dt.time

    # Sort by timestamp
    temp_df = temp_df.sort_values(by='Timestamp')

    plt.figure(figsize=(10, 5))
    sns.histplot(data=temp_df, x='Timestamp', hue='Sender', multiple='stack', shrink=0.8)
    plt.title('Message Distribution Over Time')
    plt.xlabel('Timestamp (HH:MM:SS)')
    plt.ylabel('Message Count')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("The DataFrame 'df' is empty or not defined. Please interact with the chat client first!")

The DataFrame 'df' is empty or not defined. Please interact with the chat client first!


In [26]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

if 'df' in globals() and not df.empty:
    # Combine all messages into a single string
    text = " ".join(df['Message'].astype(str).tolist())

    # Generate the Word Cloud
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='#121212',
        colormap='viridis',
        max_words=100
    ).generate(text)

    # Plot the Word Cloud
    plt.figure(figsize=(10, 5), facecolor='#121212')
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Chat Messages', color='#ffffff', fontsize=16, pad=15)
    plt.tight_layout(pad=0)
    plt.show()
else:
    print("The DataFrame 'df' is empty or not defined. Please interact with the chat client first!")

The DataFrame 'df' is empty or not defined. Please interact with the chat client first!


In [27]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'df' in globals() and not df.empty:
    # Ensure the Message_Length column is calculated
    df['Message_Length'] = df['Message'].astype(str).apply(len)

    # Generate a box plot with explicit hue to address deprecation warnings
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x='Sender', y='Message_Length', hue='Sender', palette='Set2', legend=False)
    sns.stripplot(data=df, x='Sender', y='Message_Length', color='black', alpha=0.3, jitter=0.1)
    plt.title('Comparison of Message Lengths by Sender')
    plt.xlabel('Sender')
    plt.ylabel('Message Length (Characters)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("The DataFrame 'df' is empty or not defined. Please interact with the chat client first!")

The DataFrame 'df' is empty or not defined. Please interact with the chat client first!


In [28]:
if 'df' in globals() and not df.empty:
    export_to_csv(df, output_filename='chat_history_export.csv')
else:
    print("No active chat history found in the DataFrame 'df' to export.")

No active chat history found in the DataFrame 'df' to export.


In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

if 'df' in globals() and not df.empty:
    # Clean and split the messages into individual words
    all_words = []
    for msg in df['Message'].astype(str):
        # Lowercase and keep only alphanumeric characters and spaces
        cleaned_msg = re.sub(r'[^a-zA-Z0-9\s]', '', msg.lower())
        all_words.extend(cleaned_msg.split())

    # Filter out empty strings if any
    all_words = [word for word in all_words if word.strip()]

    # Count the word frequencies
    word_counts = Counter(all_words)
    word_freq_df = pd.DataFrame(word_counts.most_common(10), columns=['Word', 'Frequency'])

    print("### Top Word Frequencies in Chat ###")
    display(word_freq_df)

    # Plot the top word frequencies
    plt.figure(figsize=(8, 4))
    sns.barplot(data=word_freq_df, x='Frequency', y='Word', hue='Word', palette='mako', legend=False)
    plt.title('Most Frequent Words in Chat')
    plt.xlabel('Occurrence Count')
    plt.ylabel('Word')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("The DataFrame 'df' is empty or not defined. Please interact with the chat client first!")

The DataFrame 'df' is empty or not defined. Please interact with the chat client first!


In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'word_freq_df' in globals() and not word_freq_df.empty:
    plt.figure(figsize=(10, 5))
    # swap x and y axes for a horizontal bar chart
    sns.barplot(data=word_freq_df, x='Frequency', y='Word', hue='Word', palette='mako', legend=False)
    plt.title('Word Frequency Distribution (Horizontal Bar Chart)')
    plt.xlabel('Frequency')
    plt.ylabel('Word')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("The DataFrame 'word_freq_df' is not defined or is empty. Please ensure it is populated first.")

The DataFrame 'word_freq_df' is not defined or is empty. Please ensure it is populated first.


In [31]:
if 'df' in globals() and not df.empty:
    if 'export_to_csv' in globals():
        export_to_csv(df, output_filename='chat_history_export.csv')
    else:
        df.to_csv('chat_history_export.csv', index=False)
        print("Successfully exported df to 'chat_history_export.csv'!")
else:
    print("The DataFrame 'df' is empty or not defined. Please interact with the chat client above first.")

The DataFrame 'df' is empty or not defined. Please interact with the chat client above first.


In [32]:
import os

file_name = 'chat_history_backup.json'
if os.path.exists(file_name):
    print(f"Yes! '{file_name}' exists in your current files.")
else:
    print(f"No, '{file_name}' does not exist yet. Please send some messages in the chat client and click 'Clear' to generate it.")

No, 'chat_history_backup.json' does not exist yet. Please send some messages in the chat client and click 'Clear' to generate it.


In [33]:
import pandas as pd
import os

file_path = 'chat_history_backup.json'

if os.path.exists(file_path):
    # Load the JSON backup into a pandas DataFrame
    df = pd.read_json(file_path)
    if not df.empty and len(df.columns) == 3:
        df.columns = ['Timestamp', 'Sender', 'Message']
    print("Successfully loaded 'chat_history_backup.json' into the DataFrame 'df'!")
    display(df.head())
else:
    print(f"The file '{file_path}' does not exist yet. Please send some messages in the chat client above and click 'Clear' to generate it first.")

The file 'chat_history_backup.json' does not exist yet. Please send some messages in the chat client above and click 'Clear' to generate it first.


In [34]:
import os

# List files using Python
files = os.listdir('.')
print("Files in current directory (Python):", files)

Files in current directory (Python): ['.config', 'sample_data']


In [35]:
# List files using a shell command
!ls -la

total 16
drwxr-xr-x 1 root root 4096 Oct  2 13:50 .
drwxr-xr-x 1 root root 4096 Oct 10 14:08 ..
drwxr-xr-x 4 root root 4096 Oct  2 13:50 .config
drwxr-xr-x 1 root root 4096 Oct  2 13:50 sample_data


In [36]:
import json
import datetime
import pandas as pd

# Create some realistic sample chat history
sample_messages = [
    ("10:00:05", "Alice", "Hello everyone! Let's start the project discussion."),
    ("10:00:20", "Client-Bot", "Received: 'Hello everyone! Let's start the project discussion.'"),
    ("10:01:15", "Bob", "Hi Alice, I am ready. What is our first milestone?"),
    ("10:01:30", "Client-Bot", "Received: 'Hi Alice, I am ready. What is our first milestone?'"),
    ("10:02:10", "Alice", "First milestone is data collection and preparation."),
    ("10:02:25", "Client-Bot", "Received: 'First milestone is data collection and preparation.'"),
    ("10:03:02", "Charlie", "I can help with data collection. I have some sample scripts."),
    ("10:03:15", "Client-Bot", "Received: 'I can help with data collection. I have some sample scripts.'"),
    ("10:04:40", "Bob", "Great! Let's set up a shared repository for that."),
    ("10:04:55", "Client-Bot", "Received: 'Great! Let's set up a shared repository for that.'")
]

# Save mock data to JSON to simulate the backup step
backup_filename = 'chat_history_backup.json'
with open(backup_filename, 'w') as f:
    json.dump(sample_messages, f, indent=4)
print(f"Mock data successfully saved to {backup_filename}")

# Load into global df and set columns for the downstream analytics cells
df = pd.DataFrame(sample_messages, columns=['Timestamp', 'Sender', 'Message'])
print("Sample DataFrame 'df' successfully loaded into memory:")
display(df)

Mock data successfully saved to chat_history_backup.json
Sample DataFrame 'df' successfully loaded into memory:


,Timestamp,Sender,Message
0,10:00:05,Alice,Hello everyone! Let's start the project discus...
1,10:00:20,Client-Bot,Received: 'Hello everyone! Let's start the pro...
2,10:01:15,Bob,"Hi Alice, I am ready. What is our first milest..."
3,10:01:30,Client-Bot,"Received: 'Hi Alice, I am ready. What is our f..."
4,10:02:10,Alice,First milestone is data collection and prepara...
5,10:02:25,Client-Bot,Received: 'First milestone is data collection ...
6,10:03:02,Charlie,I can help with data collection. I have some s...
7,10:03:15,Client-Bot,Received: 'I can help with data collection. I ...
8,10:04:40,Bob,Great! Let's set up a shared repository for that.
9,10:04:55,Client-Bot,Received: 'Great! Let's set up a shared reposi...


In [37]:
# Calculate message frequency per sender
if 'df' in globals() and not df.empty:
    frequency = df['Sender'].value_counts().reset_index()
    frequency.columns = ['Sender', 'Message Count']
    display(frequency)
else:
    print("The DataFrame 'df' is not defined or is empty. Please run the cell above to generate mock data first.")

,Sender,Message Count
0,Client-Bot,5
1,Alice,2
2,Bob,2
3,Charlie,1
